# Tratamento Inicial dos Dados Brutos de Qualidade da Água

## 1. Introdução

Este documento descreve os procedimentos metodológicos adotados para o tratamento inicial dos dados brutos de qualidade da água fornecidos pelo Instituto Estadual do Ambiente (INEA).  

O objetivo desta etapa é transformar os dados originais, que apresentam inconsistências estruturais e redundâncias, em uma base consolidada, padronizada e tecnicamente adequada para análises estatísticas posteriores.

O período analisado compreende os anos de **2012 a 2023**, conforme disponibilidade na base original.

## 2. Conceito de Limite de Detecção (LD)

O Limite de Detecção (LD) representa o menor valor que pode ser mensurado com confiabilidade pelo método analítico empregado.

Quando um valor real encontra-se abaixo desse limite, o resultado pode ser reportado da seguinte forma:

- A coluna da variável apresenta o valor mínimo detectável.
- A coluna de LD apresenta um indicador, como "<".

Exemplo conceitual:

Se o equipamento mede pH apenas entre 6 e 10 e o valor real for 5.9:

- A coluna `pH` poderá registrar 6.
- A coluna `LD` associada apresentará "<".

Isso indica que o valor verdadeiro é inferior ao limite analítico.

Esse tipo de dado é classificado como **censurado à esquerda**, sendo relevante para análises estatísticas específicas.

## 3. Estruturação do DataFrame Final

O DataFrame final foi estruturado da seguinte forma:

1. Colunas identificadoras:
   - Local (estação de monitoramento)
   - Data (data da coleta)

2. Para cada variável de interesse:
   - Coluna de Limite de Detecção (_LD)
   - Coluna consolidada da variável

## 4. Resultado do Processo de Tratamento

Ao final desta etapa, a base de dados apresenta as seguintes características:

- Ausência de colunas redundantes.
- Variáveis padronizadas e consolidadas.
- Correspondência explícita entre variável e seu respectivo LD.
- Estrutura adequada para análises estatísticas e ambientais subsequentes.

## Utils

### Importações

In [1]:
import re
import unicodedata
import pandas as pd
import openpyxl as px
from pathlib import Path

### Configurações

In [2]:
INFORMATION_SHEET_NAME = 'Estações 2012 a 2024' # Aba com as informações das estações
INFORMATION_SHEET_COLUMNS = ['CÓDIGO ESTAÇÃO RESUMIDO', 'CÓDIGO ESTAÇÃO', 'LATITUDE', 'LONGITUDE', 'CORPO D\'ÁGUA'] # Colunas de interesse da aba de informações das estações

WATER_INFORMATION_BASE_COLUMNS = ['Local', 'Codigo Local', 'Data'] # Colunas básicas de interesse dos dados de qualidade da água
WATER_INFORMATION_INTEREST_COLUMNS = ['DBO', 'OD', 'Nitrato', 'Nitrogênio Amoniacal Total', 'Fósforo Total', 'Condutividade', 'pH', 'Turbidez', 'Temperatura da Água', 'Sólidos Suspensos Totais', 'Coliformes Termotolerantes', 'Cianobacterias', 'Microcistinas'] # Colunas de interesse dos dados de qualidade da água
INTEREST_COLUMN_MATCH_LD = {'DBO': 'LD', 'OD': 'LD.2', 'Nitrato': 'LD.3', 'Nitrogênio Amoniacal Total': 'LD.5', 'Fósforo Total': 'LD.6', 'Condutividade': 'LD.8', 'pH': 'LD.9', 'Turbidez': 'LD.10', 'Temperatura da Água': 'LD.13', 'Sólidos Suspensos Totais': 'LD.15', 'Coliformes Termotolerantes': 'LD.40', 'Cianobacterias': 'LD.42'} # Mapeamento entre as colunas de interesse e suas respectivas colunas de limite de detecção
LD_COLUMNS_TO_REMOVE = ['LD.1', 'LD.16', 'LD.17', 'LD.19', 'LD.20', 'LD.21', 'LD.22', 'LD.23', 'LD.24', 'LD.25', 'LD.26', 'LD.27', 'LD.28', 'LD.29', 'LD.30', 'LD.31', 'LD.32', 'LD.33', 'LD.34', 'LD.36', 'LD.37', 'LD.38', 'LD.39'] # Colunas de limite de detecção a serem removidas

CODIGO_ESTACOES_RESUMIDO = ['JC341', 'JC342', 'CM320', 'MR361', 'MR363', 'MR369', 'TJ303', 'TJ306'] # Códigos resumidos das estações de interesse
CODIGO_ESTACOES = ['01RJ20CM0320', '01RJ20JC0341', '01RJ20JC0342', '01RJ20MR0361', '01RJ20MR0363', '01RJ20MR0369', '01RJ20TJ0303', '01RJ20TJ0306'] # Códigos das estações de interesse

DATA_PATH = Path("../../Data/RawData/WaterQualityRawData.xlsx") # Caminho para o arquivo de dados brutos
AVALIABLE_YEARS = ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'] # Anos disponíveis no arquivo de dados brutos

### Funções auxiliares

In [3]:
# Função para padronizar os nomes das colunas, removendo acentos, convertendo para minúsculas e removendo parênteses e espaços extras
def canonical_name(col):
    col = unicodedata.normalize('NFKD', col).encode('ASCII', 'ignore').decode('ASCII')
    col = col.lower()
    col = re.sub(r'\(.*?\)', '', col)
    col = re.sub(r'\s+', ' ', col).strip()
    return col

## Código

### Aquisição e Consolidação Inicial dos Dados

Os dados foram obtidos a partir de planilhas anuais disponibilizadas pelo INEA.  

As etapas iniciais consistiram em:

1. Leitura de todos os arquivos anuais disponíveis.
2. Consolidação dos dados em um único DataFrame.
3. Filtragem dos registros com base nos códigos das estações de monitoramento de interesse.

Essa etapa garantiu que apenas os pontos amostrais relevantes para o estudo fossem considerados nas análises subsequentes.

In [4]:
# Pega as informações das estações, pegando apenas as estações de interesse
df_estacoes = pd.read_excel(DATA_PATH, sheet_name=INFORMATION_SHEET_NAME)

df_estacoes = df_estacoes[df_estacoes['CÓDIGO ESTAÇÃO RESUMIDO'].isin(CODIGO_ESTACOES_RESUMIDO)]
df_estacoes[INFORMATION_SHEET_COLUMNS]

,CÓDIGO ESTAÇÃO RESUMIDO,CÓDIGO ESTAÇÃO,LATITUDE,LONGITUDE,CORPO D'ÁGUA
202,CM320,01RJ20CM0320,-22.976072,-43.365167,Lagoa de Camorim
203,JC341,01RJ20JC0341,-22.979856,-43.401658,Lagoa de Jacarepaguá
204,JC342,01RJ20JC0342,-22.976275,-43.381347,Lagoa de Jacarepaguá
205,MR361,01RJ20MR0361,-23.018333,-43.435556,Lagoa de Marapendi
206,MR363,01RJ20MR0363,-23.010133,-43.392403,Lagoa de Marapendi
207,MR369,01RJ20MR0369,-23.006606,-43.365833,Lagoa de Marapendi
208,TJ303,01RJ20TJ0303,-23.007478,-43.303072,Lagoa da Tijuca
209,TJ306,01RJ20TJ0306,-22.998617,-43.318550,Lagoa da Tijuca


In [5]:
# Para cada ano disponível no excel, pegamos os dados de qualidade da água, filtramos apenas as estações de interesse e concatenamos em um único dataframe
df_water_data = pd.DataFrame()

for year in AVALIABLE_YEARS:
    df_year = pd.read_excel(DATA_PATH, sheet_name=year)
    df_year = df_year[df_year['Local'].isin(CODIGO_ESTACOES)]
    df_water_data = pd.concat([df_water_data, df_year], ignore_index=True)
    
df_water_data = df_water_data.merge(df_estacoes[['CÓDIGO ESTAÇÃO RESUMIDO', 'CÓDIGO ESTAÇÃO', 'CORPO D\'ÁGUA']], left_on='Local', right_on='CÓDIGO ESTAÇÃO').drop(columns=['CÓDIGO ESTAÇÃO', 'Local']).rename(columns={'CÓDIGO ESTAÇÃO RESUMIDO': 'Codigo Local', 'CORPO D\'ÁGUA': 'Local'})

display(df_water_data)

,Data,Hora,LD,DBO (mg/L),LD.1,DQO (mg/L),LD.2,OD (mg/L),LD.3,Nitrato (mg/L),...,Cromo total (mg/L),Ferro dissolvido (mg/L),Manganês total (mg/L),Níquel total (m/L),Zinco total (mg/L),Cianotoxinas / Microcistinas,Óleos e graxas (mg/L),Transparencia (m),Codigo Local,Local
0,2012-01-12 00:00:00,1900-01-01 07:35:28,NaN,14.0,NaN,NaN,NaN,4.8,NaN,0.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CM320,Lagoa de Camorim
1,2012-02-09 00:00:00,1900-01-01 09:00:28,NaN,40.0,NaN,NaN,NaN,0.4,<,0.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CM320,Lagoa de Camorim
2,2012-03-08 00:00:00,1900-01-01 06:45:28,NaN,32.0,NaN,NaN,NaN,0.0,NaN,0.07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CM320,Lagoa de Camorim
3,2012-04-12 00:00:00,1900-01-01 07:10:28,NaN,20.0,NaN,NaN,NaN,3.2,NaN,0.07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CM320,Lagoa de Camorim
4,2012-05-17 00:00:00,1900-01-01 08:55:28,NaN,16.0,NaN,NaN,NaN,3.4,NaN,0.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CM320,Lagoa de Camorim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,2023-08-15 00:00:00,1900-01-01 11:41:28,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.278,NaN,NaN,TJ303,Lagoa da Tijuca
563,2023-10-18 00:00:00,1900-01-01 10:40:28,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.491,NaN,NaN,TJ303,Lagoa da Tijuca
564,2023-12-13 00:00:00,1900-01-01 10:30:28,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.099,NaN,NaN,TJ303,Lagoa da Tijuca
565,2023-02-27 00:00:00,1900-01-01 10:12:28,NaN,28.0,NaN,NaN,NaN,NaN,NaN,0.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TJ306,Lagoa da Tijuca


### Remoção de Colunas Integralmente Vazias

Foi realizada a identificação e remoção de:

- Colunas cujos registros estavam integralmente vazios.
- Suas respectivas colunas associadas de Limite de Detecção (LD).

A exclusão dessas colunas reduz ruído estrutural e melhora a eficiência computacional das etapas posteriores.

In [6]:
# Removendo colunas vazias e colunas de limite de detecção dessas colunas vazias
cols_empty_to_drop = [
    col for col in df_water_data.columns
    if (
        not col.startswith("LD") and
        df_water_data[col]
            .replace(r'^\s*$', pd.NA, regex=True)
            .isna()
            .all()
    )
]

cols_to_drop = cols_empty_to_drop + LD_COLUMNS_TO_REMOVE

cols_to_drop = [col for col in cols_to_drop if col in df_water_data.columns]

df_water_data.drop(columns=cols_to_drop, inplace=True)

print(f"{len(cols_to_drop)} colunas removidas.")

86 colunas removidas.


### Identificação de Inconsistências Estruturais

Durante a inspeção exploratória da base de dados, foram identificadas inconsistências estruturais, incluindo:

- Colunas duplicadas representando a mesma variável.
- Diferenças de nomenclatura decorrentes de:
  - Espaços adicionais.
  - Diferenças entre letras maiúsculas e minúsculas.
  - Presença ou ausência de unidades no nome da coluna.
  - Variações na acentuação.

Exemplos observados:

- "pH" e "pH  "
- "OD (mg/L)" e "OD  (mg/L)"
- "Nitrogênio amoniacal total (mg/L)" e "Nitrogênio Amoniacal Total (mg/L)"

Essas inconsistências poderiam comprometer análises automatizadas e gerar duplicidade indevida de variáveis.

### Padronização de Nomenclatura

Foi implementado um processo sistemático de padronização para:

- Remover acentuação.
- Eliminar unidades descritas entre parênteses.
- Normalizar espaços múltiplos.
- Uniformizar caixa de texto (lowercase).

Essa abordagem permitiu identificar colunas semanticamente equivalentes, independentemente de variações superficiais de escrita.

### Consolidação de Variáveis Duplicadas

Após a padronização, variáveis equivalentes foram consolidadas em colunas únicas.

A regra aplicada foi:

- Se apenas uma coluna possuía valor no registro → esse valor foi mantido.
- Se múltiplas colunas possuíam valores no mesmo registro → foi calculada a média aritmética linha a linha.
- Se todas estavam ausentes → o valor permaneceu como ausente.

Essa estratégia assegura:

- Coerência estatística.
- Preservação da informação disponível.
- Eliminação de redundâncias estruturais.

In [7]:
# Renomeando as colunas de cianotoxinas/microcistinas pois elas estão com o mesmo nome, mas medidas diferentes. Então 
df_water_data.rename(columns={
    'Cianotoxinas/Microcistinas (cél/L)': 'Cianobacterias',
    'Cianotoxinas/Microcistinas (µg/L)': 'Microcistinas',
}, inplace=True)

In [8]:
# Dicionário para mapear os nomes originais das colunas para seus nomes canônicos
normalized_columns = {col: canonical_name(col) for col in df_water_data.columns}

df_final = pd.DataFrame()

# Iniciamos o dataframe final apenas com as colunas básicas de interesse
for base_col in WATER_INFORMATION_BASE_COLUMNS:
    base_canonical = canonical_name(base_col)
    
    matching_cols = [
        original
        for original, norm in normalized_columns.items()
        if norm == base_canonical
    ]
    
    if matching_cols:
        df_final[base_col] = df_water_data[matching_cols[0]]

In [9]:
# Para as variáveis numéricas, vamos procurar por todas as colunas que correspondem à variável de interesse (usando o nome canônico) e tirar a média entre elas. 
# Além disso, se a variável tiver uma coluna de limite de detecção (LD), vamos adicioná-la ao dataframe final usando o nome da configuração.
for var in WATER_INFORMATION_INTEREST_COLUMNS:
    var_canonical = canonical_name(var)
    
    matching_cols = [
        original
        for original, norm in normalized_columns.items()
        if norm == var_canonical
    ]
    
    if not matching_cols:
        continue
    
    temp = df_water_data[matching_cols].apply(pd.to_numeric, errors='coerce')
    consolidated = temp.mean(axis=1)
    
    if var in INTEREST_COLUMN_MATCH_LD:
        ld_col = INTEREST_COLUMN_MATCH_LD[var]
        
        if ld_col in df_water_data.columns:
            df_final[f"{var}_LD"] = df_water_data[ld_col]
    
    df_final[var] = consolidated

In [10]:
df_final.head()

,Local,Codigo Local,Data,DBO_LD,DBO,OD_LD,OD,Nitrato_LD,Nitrato,Nitrogênio Amoniacal Total_LD,...,Turbidez,Temperatura da Água_LD,Temperatura da Água,Sólidos Suspensos Totais_LD,Sólidos Suspensos Totais,Coliformes Termotolerantes_LD,Coliformes Termotolerantes,Cianobacterias_LD,Cianobacterias,Microcistinas
0,Lagoa de Camorim,CM320,2012-01-12 00:00:00,NaN,14.0,NaN,4.8,NaN,0.04,NaN,...,136.0,NaN,26.0,NaN,60.0,NaN,23000.0,NaN,NaN,NaN
1,Lagoa de Camorim,CM320,2012-02-09 00:00:00,NaN,40.0,NaN,0.4,<,0.01,NaN,...,2.3,NaN,NaN,NaN,38.0,NaN,2000.0,NaN,0.401,NaN
2,Lagoa de Camorim,CM320,2012-03-08 00:00:00,NaN,32.0,NaN,0.0,NaN,0.07,NaN,...,29.0,NaN,22.0,NaN,53.0,>,160000.0,NaN,36.800,NaN
3,Lagoa de Camorim,CM320,2012-04-12 00:00:00,NaN,20.0,NaN,3.2,NaN,0.07,NaN,...,90.0,NaN,27.0,NaN,83.0,NaN,240000.0,NaN,47.340,NaN
4,Lagoa de Camorim,CM320,2012-05-17 00:00:00,NaN,16.0,NaN,3.4,NaN,0.01,NaN,...,35.0,NaN,22.0,NaN,39.0,NaN,54000.0,NaN,NaN,NaN


In [11]:
df_final.to_excel("../../Data/IntermediaryData/WaterQualityInitialData.xlsx", index=False)